In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── 경로 설정 (본인 환경에 맞게 수정)
DATA_DIR = Path("./data")   # 데이터 파일 위치

# 결과 저장 폴더 생성
Path("./output").mkdir(exist_ok=True)

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [3]:
# 원천 데이터 로드
labels  = pd.read_csv(DATA_DIR / "labels.csv")
weekly  = pd.read_excel(DATA_DIR / "주간시총300.xlsx")

print("=== labels.csv ===")
print(f"  shape : {labels.shape}")
print(f"  컬럼  : {labels.columns.tolist()}")
print(f"  window: {sorted(labels['window'].unique())}")
print()
print("=== 주간시총300.xlsx ===")
print(f"  shape    : {weekly.shape}")
print(f"  컬럼     : {weekly.columns.tolist()}")
print(f"  날짜범위 : {weekly['date'].min().date()} ~ {weekly['date'].max().date()}")
print(f"  유니크 ticker: {weekly['ticker'].nunique()}")

=== labels.csv ===
  shape : (2491, 6)
  컬럼  : ['window', 'ticker', 'is_member', 'label_in', 'label_out', 'actual_rank']
  window: ['2020_H1', '2020_H2', '2021_H1', '2021_H2', '2022_H1', '2022_H2', '2023_H1', '2023_H2', '2024_H1', '2024_H2', '2025_H1', '2025_H2']

=== 주간시총300.xlsx ===
  shape    : (108000, 13)
  컬럼     : ['date', 'ticker', 'company', 'close', 'shares', '시가총액', '시가총액_rank', '유동주식수', '유통주식비율', '거래대금', '업종', '시총순위변화율', '과거지수순위']
  날짜범위 : 2019-05-03 ~ 2026-03-17
  유니크 ticker: 497


# 컬럼 rename

In [4]:
# 주간시총300.xlsx → WEEKLY_SAMPLE 딕셔너리 기준으로 rename
weekly = weekly.rename(columns={
    '시가총액'     : 'mktcap',
    '시가총액_rank': 'mktcap_rank',
    '거래대금'     : 'trading_value',
    '유동주식수'   : 'float_shares_raw',   # SHARES_LOCK 완성 전 임시
    '유통주식비율' : 'float_ratio_raw',    # SHARES_LOCK 완성 전 임시
    '업종'         : 'sector_name_raw',    # SECTOR_MAP 완성 전 임시
    '시총순위변화율': 'rank_chg_pct_raw',
    '과거지수순위' : 'past_index_rank_raw',
})

print("rename 후 컬럼:", weekly.columns.tolist())
print(weekly[['ticker','date','close','mktcap','mktcap_rank','trading_value']].head(3))

rename 후 컬럼: ['date', 'ticker', 'company', 'close', 'shares', 'mktcap', 'mktcap_rank', 'float_shares_raw', 'float_ratio_raw', 'trading_value', 'sector_name_raw', 'rank_chg_pct_raw', 'past_index_rank_raw']
  ticker       date   close           mktcap  mktcap_rank  trading_value
0   5930 2019-05-03   45300  268159597866600            1            NaN
1    660 2019-05-03   80400   57301270146000            2            NaN
2  68270 2019-05-03  166830   38545781382630            3            NaN


# 기준일 매핑 

In [5]:
# KRX 코스피200 정기변경 기준일 (해당 주 금요일 기준)
# 실제 발표일과 ±1주 오차 허용 → 가장 가까운 데이터 날짜로 snap
WINDOW_EVAL_DATE = {
    '2020_H1': '2020-06-05',
    '2020_H2': '2020-12-04',
    '2021_H1': '2021-06-04',
    '2021_H2': '2021-12-03',
    '2022_H1': '2022-06-10',
    '2022_H2': '2022-12-09',
    '2023_H1': '2023-06-09',
    '2023_H2': '2023-12-08',
    '2024_H1': '2024-06-07',
    '2024_H2': '2024-12-06',
    '2025_H1': '2025-06-06',
    # 2025_H2는 미래 → 예측 대상, 학습 제외
}
WINDOWS_ORDERED = sorted(WINDOW_EVAL_DATE.keys())

# weekly에서 실제로 존재하는 가장 가까운 날짜 찾는 함수
available_dates = sorted(weekly['date'].unique())

def get_closest_date(target_str):
    target = pd.Timestamp(target_str)
    return min(available_dates, key=lambda d: abs(d - target))

# 확인
for w, d in WINDOW_EVAL_DATE.items():
    closest = get_closest_date(d)
    print(f"{w}  목표: {d}  →  실제: {closest.date()}")

2020_H1  목표: 2020-06-05  →  실제: 2020-06-05
2020_H2  목표: 2020-12-04  →  실제: 2020-12-04
2021_H1  목표: 2021-06-04  →  실제: 2021-06-04
2021_H2  목표: 2021-12-03  →  실제: 2021-12-03
2022_H1  목표: 2022-06-10  →  실제: 2022-06-10
2022_H2  목표: 2022-12-09  →  실제: 2022-12-09
2023_H1  목표: 2023-06-09  →  실제: 2023-06-09
2023_H2  목표: 2023-12-08  →  실제: 2023-12-08
2024_H1  목표: 2024-06-07  →  실제: 2024-06-07
2024_H2  목표: 2024-12-06  →  실제: 2024-12-06
2025_H1  목표: 2025-06-06  →  실제: 2025-06-05


# window(period)별 rank 스냅샷 생성

In [6]:
snap_rows = []

for window, eval_date_str in WINDOW_EVAL_DATE.items():
    closest = get_closest_date(eval_date_str)
    snap = (
        weekly[weekly['date'] == closest]
        [['ticker', 'mktcap_rank', 'mktcap', 'trading_value']]
        .copy()
    )
    snap['window']    = window
    snap['snap_date'] = closest
    snap_rows.append(snap)

rank_snap = pd.concat(snap_rows, ignore_index=True)

print("rank_snap shape:", rank_snap.shape)
print("null 확인:")
print(rank_snap.isnull().sum())
print(rank_snap.head(5))

rank_snap shape: (3300, 6)
null 확인:
ticker              0
mktcap_rank         0
mktcap              0
trading_value    3300
window              0
snap_date           0
dtype: int64
   ticker  mktcap_rank           mktcap  trading_value   window  snap_date
0    5930            1  328539904671000            NaN  2020_H1 2020-06-05
1     660            2   64428293796000            NaN  2020_H1 2020-06-05
2   68270            3   50617821971619            NaN  2020_H1 2020-06-05
3  207940            4   44775153809505            NaN  2020_H1 2020-06-05
4    5935            5   38962790206000            NaN  2020_H1 2020-06-05


In [8]:
indiv = pd.read_excel("data/개별종목_202505_202511_.xlsx")

print("=== 개별종목_202505_202511_.xlsx ===")
print("shape:", indiv.shape)
print("columns:", indiv.columns.tolist())
print()
print(indiv.head(3).to_string())
print()
print("dtypes:\n", indiv.dtypes)
print()

# 날짜 범위
print("기준일자 unique 수:", indiv['기준일자'].nunique())
print("기준일자 range:", indiv['기준일자'].min(), "~", indiv['기준일자'].max())
print()

# ticker 형식 확인
print("종목코드 sample:", indiv['종목코드'].head(10).tolist())
print("종목코드 dtype:", indiv['종목코드'].dtype)

=== 개별종목_202505_202511_.xlsx ===
shape: (23067, 15)
columns: ['기준일자', '종목코드', '종목명', '보(0)/우(1)', '업종명', '종가', '거래대금', '시가총액', '상장주식수', 'is_member', 'label_in', 'label_out', 'actual_rank', 'Unnamed: 13', '2019-2025년 주간/코스피 내 종목 정보']

       기준일자   종목코드     종목명  보(0)/우(1)    업종명     종가        거래대금          시가총액     상장주식수  is_member  label_in  label_out  actual_rank Unnamed: 13  2019-2025년 주간/코스피 내 종목 정보
0  20250502  95570  AJ네트웍스          0  일반서비스   3760  1032650980  170150000000  45252759        NaN       NaN        NaN          NaN         NaN  kospi200에 선정된 종목에 대해서 라벨링
1  20250502   6840   AK홀딩스          0   기타금융  10620   112280110  140689000000  13247561        1.0       0.0        1.0          NaN         NaN                        NaN
2  20250502  27410     BGF          0   기타금융   3585   101444107  343145000000  95716791        1.0       0.0        1.0          NaN         NaN                        NaN

dtypes:
 기준일자                           int64
종목코드                          o

In [9]:
# zero-pad 통일
indiv['종목코드'] = indiv['종목코드'].astype(str).str.zfill(6)
indiv['기준일자'] = pd.to_datetime(indiv['기준일자'].astype(str), format='%Y%m%d')

print("=== 키 형식 비교 ===")
print("weekly  ticker 예시:", weekly['ticker'].head(5).tolist())
print("indiv   종목코드 예시:", indiv['종목코드'].head(5).tolist())
print()
print("weekly  date 예시:", weekly['date'].head(3).tolist())
print("indiv   기준일자 예시:", indiv['기준일자'].head(3).tolist())
print()

# 날짜 오버랩 확인
weekly_dates = set(weekly['date'].dt.date)
indiv_dates  = set(indiv['기준일자'].dt.date)
overlap = weekly_dates & indiv_dates
print(f"weekly 날짜 수: {len(weekly_dates)}")
print(f"indiv  날짜 수: {len(indiv_dates)}")
print(f"겹치는 날짜 수: {len(overlap)}")
print(f"겹치는 날짜들: {sorted(overlap)}")
print()

# ticker 오버랩 확인
weekly_tickers = set(weekly['ticker'].unique())
indiv_tickers  = set(indiv['종목코드'].unique())
print(f"weekly ticker 수: {len(weekly_tickers)}")
print(f"indiv  ticker 수: {len(indiv_tickers)}")
print(f"겹치는 ticker 수: {len(weekly_tickers & indiv_tickers)}")

=== 키 형식 비교 ===
weekly  ticker 예시: [5930, 660, 68270, 5935, 96770]
indiv   종목코드 예시: ['095570', '006840', '027410', '282330', '138930']

weekly  date 예시: [Timestamp('2019-05-03 00:00:00'), Timestamp('2019-05-03 00:00:00'), Timestamp('2019-05-03 00:00:00')]
indiv   기준일자 예시: [Timestamp('2025-05-02 00:00:00'), Timestamp('2025-05-02 00:00:00'), Timestamp('2025-05-02 00:00:00')]

weekly 날짜 수: 360
indiv  날짜 수: 24
겹치는 날짜 수: 24
겹치는 날짜들: [datetime.date(2025, 5, 2), datetime.date(2025, 5, 9), datetime.date(2025, 5, 16), datetime.date(2025, 5, 23), datetime.date(2025, 5, 30), datetime.date(2025, 6, 13), datetime.date(2025, 6, 20), datetime.date(2025, 6, 27), datetime.date(2025, 7, 4), datetime.date(2025, 7, 11), datetime.date(2025, 7, 18), datetime.date(2025, 7, 25), datetime.date(2025, 8, 1), datetime.date(2025, 8, 8), datetime.date(2025, 8, 22), datetime.date(2025, 8, 29), datetime.date(2025, 9, 5), datetime.date(2025, 9, 12), datetime.date(2025, 9, 19), datetime.date(2025, 9, 26), datetime.date

In [11]:
# indiv zero-pad 제거 → 숫자형 문자열로 통일
# '006840' → '6840', '00104K' 같은 우선주는 별도 처리
indiv_raw = pd.read_excel("data/개별종목_202505_202511_.xlsx")

# 우선주(알파벳 포함) 제외 여부 먼저 확인
alpha_mask = indiv_raw['종목코드'].astype(str).str.contains('[A-Za-z]', regex=True)
print(f"알파벳 포함 종목코드 수 (우선주 등): {alpha_mask.sum()}")
print(indiv_raw[alpha_mask]['종목코드'].unique()[:10])
print()

# 숫자만 있는 종목코드 → int 변환 후 str (앞 0 제거)
indiv = indiv_raw[~alpha_mask].copy()
indiv['종목코드'] = indiv['종목코드'].astype(str).str.lstrip('0')  # '006840' → '6840'
indiv['기준일자'] = pd.to_datetime(indiv['기준일자'].astype(str), format='%Y%m%d')

# weekly ticker도 str로 통일
weekly['ticker'] = weekly['ticker'].astype(str)
indiv['종목코드'] = indiv['종목코드'].astype(str)

# 재확인
weekly_sample = set(weekly[weekly['date'] >= '2025-05-01']['ticker'].unique())
indiv_sample  = set(indiv['종목코드'].unique())

print(f"2025-05 이후 weekly ticker 수: {len(weekly_sample)}")
print(f"indiv ticker 수: {len(indiv_sample)}")
print(f"겹치는 수: {len(weekly_sample & indiv_sample)}")
print()
print("weekly에만 있는 예시 5개:", list(weekly_sample - indiv_sample)[:5])
print("indiv에만 있는 예시 5개:",  list(indiv_sample - weekly_sample)[:5])

알파벳 포함 종목코드 수 (우선주 등): 567
['00104K' '37550L' '37550K' '38380K' '03473K' '28513K' '00806K' '35320K'
 '33626K' '33626L']

2025-05 이후 weekly ticker 수: 349
indiv ticker 수: 942
겹치는 수: 343

weekly에만 있는 예시 5개: ['00088K', '00680K', '0120G0', '00104K', '0126Z0']
indiv에만 있는 예시 5개: ['5010', '19680', '900140', '4960', '6740']


# weekly 알파벳 코드 확인 후 JOIN

In [12]:
# weekly에서 알파벳 포함 ticker 확인
weekly_alpha = weekly['ticker'].astype(str).str.contains('[A-Za-z]', regex=True)
print(f"weekly 알파벳 포함 ticker 수: {weekly_alpha.sum()}")
print("예시:", weekly[weekly_alpha]['ticker'].unique()[:10].tolist())
print()

# 두 데이터 모두 알파벳 포함 제외 → 순수 숫자 ticker만 사용
weekly_clean = weekly[~weekly_alpha].copy()
weekly_clean['ticker'] = weekly_clean['ticker'].astype(str)

# indiv에서 trading_value만 추출해서 weekly에 보완
indiv_tv = indiv[['기준일자', '종목코드', '거래대금']].copy()
indiv_tv.columns = ['date', 'ticker', 'trading_value_indiv']

# LEFT JOIN: weekly_clean 기준으로 indiv trading_value 붙이기
weekly_clean = weekly_clean.merge(
    indiv_tv,
    on=['date', 'ticker'],
    how='left'
)

# trading_value 보완: weekly 원본이 null이면 indiv 값으로 채움
weekly_clean['trading_value'] = weekly_clean['trading_value'].fillna(
    weekly_clean['trading_value_indiv']
)
weekly_clean = weekly_clean.drop(columns=['trading_value_indiv'])

# 결과 확인
print("=== trading_value 보완 결과 ===")
tv_null = weekly_clean[weekly_clean['date'] >= '2025-05-01']['trading_value'].isna().sum()
tv_total = weekly_clean[weekly_clean['date'] >= '2025-05-01'].shape[0]
print(f"2025-05 이후 trading_value null: {tv_null} / {tv_total}")
print()
tv_null_all = weekly_clean['trading_value'].isna().sum()
tv_total_all = weekly_clean.shape[0]
print(f"전체 trading_value null: {tv_null_all} / {tv_total_all}")
print(f"  → 2025년 이전 구간은 원래 없으므로 정상")
print()
print(weekly_clean[weekly_clean['date'] >= '2025-05-01'][
    ['date','ticker','mktcap_rank','trading_value']
].head(5))

weekly 알파벳 포함 ticker 수: 353
예시: ['00680K', '00279K', '00088K', '00104K', '0126Z0', '0120G0']

=== trading_value 보완 결과 ===
2025-05 이후 trading_value null: 6811 / 13959

전체 trading_value null: 100499 / 107647
  → 2025년 이전 구간은 원래 없으므로 정상

            date  ticker  mktcap_rank  trading_value
93688 2025-05-02    5930            1   1.227730e+12
93689 2025-05-02     660            2   7.130780e+11
93690 2025-05-02  373220            3   6.530201e+10
93691 2025-05-02  207940            4   7.949472e+10
93692 2025-05-02  329180            5   6.785101e+10


In [13]:
# 2025-05 이후 구간만 분리
recent = weekly_clean[weekly_clean['date'] >= '2025-05-01'].copy()

# null인 행만 추출
null_tv = recent[recent['trading_value'].isna()]

print(f"null 행 수: {len(null_tv)}")
print(f"null ticker 수 (unique): {null_tv['ticker'].nunique()}")
print(f"null 날짜 수 (unique): {null_tv['date'].nunique()}")
print()

# 날짜별 null 수
print("=== 날짜별 null 수 ===")
print(null_tv.groupby('date').size().to_string())
print()

# null ticker 중 indiv에 있는지 확인
null_tickers = set(null_tv['ticker'].unique())
indiv_tickers = set(indiv['종목코드'].unique())
print(f"null ticker 중 indiv에 있는 수: {len(null_tickers & indiv_tickers)}")
print(f"null ticker 중 indiv에 없는 수: {len(null_tickers - indiv_tickers)}")
print()
print("indiv에 없는 null ticker 예시 10개:")
print(list(null_tickers - indiv_tickers)[:10])

null 행 수: 6811
null ticker 수 (unique): 332
null 날짜 수 (unique): 23

=== 날짜별 null 수 ===
date
2025-06-05    298
2025-08-14    298
2025-10-02    297
2025-11-07    297
2025-11-14    297
2025-11-21    297
2025-11-28    296
2025-12-05    296
2025-12-12    296
2025-12-19    296
2025-12-26    296
2026-01-02    296
2026-01-09    296
2026-01-16    296
2026-01-23    295
2026-01-30    295
2026-02-06    295
2026-02-13    295
2026-02-20    296
2026-02-27    295
2026-03-06    296
2026-03-13    296
2026-03-17    296

null ticker 중 indiv에 있는 수: 331
null ticker 중 indiv에 없는 수: 1

indiv에 없는 null ticker 예시 10개:
['279570']


## 진단결과
indiv 보유 날짜: 2025-05-02 ~ 2025-10-31  (24개)
null 발생 날짜: 2025-06-05, 2025-08-14 → indiv에 없는 날짜
              2025-11-07 이후 → indiv 범위 밖

## 결론
- trading_value 베이스라인에서 제외
trading_value는 2025-05 이전 구간 전체 + 2025-11 이후 전체가 null이에요. 학습 데이터 대부분(2020_H1 ~ 2024_H2)에 값이 없으니 베이스라인 피처로 쓸 수가 없어요.
지금 당장 할 것: trading_value는 베이스라인에서 제외하고 진행해요. 나중에 FEATURE_KRX 전체 피처 만들 때 avg_trading_value(6개월 평균)로 제대로 만들면 돼요.

In [14]:
# weekly_clean에서 trading_value는 보관만 하고 피처로는 안 씀
# (나중에 avg_trading_value 만들 때 사용)
print("=== 최종 weekly_clean 상태 ===")
print("shape:", weekly_clean.shape)
print("columns:", weekly_clean.columns.tolist())
print()

# rank_snap도 weekly_clean 기준으로 재생성
# (알파벳 ticker 제거된 버전으로 다시 만들어야 함)
snap_rows = []
for window, eval_date_str in WINDOW_EVAL_DATE.items():
    closest = get_closest_date(eval_date_str)
    snap = (
        weekly_clean[weekly_clean['date'] == closest]
        [['ticker', 'mktcap_rank', 'mktcap']]
        .copy()
    )
    snap['window']    = window
    snap['snap_date'] = closest
    snap_rows.append(snap)

rank_snap = pd.concat(snap_rows, ignore_index=True)

print("rank_snap 재생성 완료")
print("shape:", rank_snap.shape)
print("null 확인:\n", rank_snap.isnull().sum())
print()
print(rank_snap.head(3))

=== 최종 weekly_clean 상태 ===
shape: (107647, 13)
columns: ['date', 'ticker', 'company', 'close', 'shares', 'mktcap', 'mktcap_rank', 'float_shares_raw', 'float_ratio_raw', 'trading_value', 'sector_name_raw', 'rank_chg_pct_raw', 'past_index_rank_raw']

rank_snap 재생성 완료
shape: (3292, 5)
null 확인:
 ticker         0
mktcap_rank    0
mktcap         0
window         0
snap_date      0
dtype: int64

  ticker  mktcap_rank           mktcap   window  snap_date
0   5930            1  328539904671000  2020_H1 2020-06-05
1    660            2   64428293796000  2020_H1 2020-06-05
2  68270            3   50617821971619  2020_H1 2020-06-05


📌 왜 2019~2024 데이터도 필요한가
2025년 12월 예측이 목표지만, ML 모델은 과거 패턴으로 학습해야 해요.
학습 데이터 (과거 패턴 학습)        예측 대상
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2020_H1 ~ 2025_H1                  2025_H2
  ↑                                   ↑
"이런 피처면 편입됐다"를 학습     "이 종목들이 편입될까?"
개별종목_202505_202511_.xlsx는 예측 시점의 피처 데이터예요. 2025-05~10 데이터로 2025_H2 시점 피처를 만들고, 그걸 학습된 모델에 넣어서 예측하는 거예요.
즉 두 데이터의 역할이 달라요.
데이터역할주간시총300.xlsx (2019~2025)학습용 피처 + 정답개별종목_202505_202511_.xlsx예측용 피처 (2025_H2 입력)

In [15]:
TRAIN_WINDOWS = [w for w in WINDOWS_ORDERED if w != '2025_H2']

# labels에서 학습 window만
df = labels[labels['window'].isin(TRAIN_WINDOWS)].copy()
df['ticker'] = df['ticker'].astype(str)

# rank_snap 병합
df = df.merge(rank_snap, on=['ticker', 'window'], how='left')

print("병합 후 shape:", df.shape)
print("mktcap_rank null:", df['mktcap_rank'].isna().sum())
print()
print("label_in 분포:")
print(df['label_in'].value_counts())
print(f"\n편입 비율: {df['label_in'].mean()*100:.1f}%")

병합 후 shape: (2283, 9)
mktcap_rank null: 106

label_in 분포:
label_in
0    2200
1      83
Name: count, dtype: int64

편입 비율: 3.6%


# was_member 피처 생성 

In [16]:
# ticker별 window 순서대로 정렬 후 is_member를 1칸 shift
df = df.sort_values(['ticker', 'window']).reset_index(drop=True)

df['was_member'] = (
    df.groupby('ticker')['is_member']
    .shift(1)
    .fillna(0)
    .astype(int)
)

print("was_member 생성 완료")
print(df[['ticker','window','is_member','was_member','label_in']].head(12).to_string())
print()
print("was_member 분포:", df['was_member'].value_counts().to_dict())

was_member 생성 완료
   ticker   window  is_member  was_member  label_in
0     100  2020_H1          1           0         0
1     100  2020_H2          1           1         0
2     100  2021_H1          1           1         0
3     100  2021_H2          1           1         0
4     100  2022_H1          1           1         0
5     100  2022_H2          1           1         0
6     100  2023_H1          1           1         0
7     100  2023_H2          1           1         0
8     100  2024_H1          1           1         0
9     100  2024_H2          1           1         0
10    100  2025_H1          1           1         0
11  10060  2020_H1          1           0         0

was_member 분포: {1: 1935, 0: 348}


# rank_momentum_1w 피처 생성

In [17]:
# weekly_clean에서 1주 전 대비 순위 변화 계산
weekly_clean = weekly_clean.sort_values(['ticker', 'date'])

weekly_clean['rank_momentum_1w'] = (
    weekly_clean.groupby('ticker')['mktcap_rank']
    .transform(lambda x: x.shift(1) - x)  # 양수 = 순위 상승
)

# window 평가 기준일 시점 값만 스냅샷
mom_rows = []
for window, eval_date_str in WINDOW_EVAL_DATE.items():
    closest = get_closest_date(eval_date_str)
    sub = (
        weekly_clean[weekly_clean['date'] == closest]
        [['ticker', 'rank_momentum_1w']]
        .copy()
    )
    sub['window'] = window
    mom_rows.append(sub)

momentum_snap = pd.concat(mom_rows, ignore_index=True)

# df에 병합
df = df.merge(momentum_snap, on=['ticker', 'window'], how='left')
df['rank_momentum_1w'] = df['rank_momentum_1w'].fillna(0)

print("rank_momentum_1w 생성 완료")
print(df[['ticker','window','mktcap_rank','rank_momentum_1w','was_member','label_in']].head(8).to_string())
print()
print("rank_momentum_1w 기초 통계:")
print(df['rank_momentum_1w'].describe().round(2))

rank_momentum_1w 생성 완료
  ticker   window  mktcap_rank  rank_momentum_1w  was_member  label_in
0    100  2020_H1         69.0              -1.0           0         0
1    100  2020_H2         70.0              -4.0           1         0
2    100  2021_H1         77.0              -3.0           1         0
3    100  2021_H2         81.0              -1.0           1         0
4    100  2022_H1         85.0              -4.0           1         0
5    100  2022_H2         74.0               1.0           1         0
6    100  2023_H1         75.0              -1.0           1         0
7    100  2023_H2         72.0               5.0           1         0

rank_momentum_1w 기초 통계:
count    2283.00
mean        0.02
std         4.51
min       -23.00
25%        -2.00
50%         0.00
75%         1.00
max        56.00
Name: rank_momentum_1w, dtype: float64


## mktcap_rank 결측치 106개 처리 

In [18]:
print("=== null 현황 ===")
print(df[['mktcap_rank', 'rank_momentum_1w', 'was_member']].isnull().sum())
print()

# mktcap_rank null인 행 확인
null_rows = df[df['mktcap_rank'].isna()]
print(f"mktcap_rank null 행 수: {len(null_rows)}")
print(f"해당 window 분포:\n{null_rows['window'].value_counts()}")
print(f"해당 label_in 분포:\n{null_rows['label_in'].value_counts()}")
print()

# 처리: mktcap_rank null = weekly에 없는 종목
# → 시총 300위 밖이거나 당시 비상장 → 학습에서 제외
df_clean = df.dropna(subset=['mktcap_rank']).copy()

print(f"제거 후 shape: {df_clean.shape}")
print(f"label_in 분포: {df_clean['label_in'].value_counts().to_dict()}")
print(f"편입 비율: {df_clean['label_in'].mean()*100:.1f}%")
print()

# 최종 피처 목록 확정
FEATURES = ['mktcap_rank', 'was_member', 'rank_momentum_1w']
TARGET   = 'label_in'

print("=== 최종 학습 데이터 ===")
print(df_clean[FEATURES + [TARGET, 'window', 'ticker']].describe().round(2))

=== null 현황 ===
mktcap_rank         106
rank_momentum_1w      0
was_member            0
dtype: int64

mktcap_rank null 행 수: 106
해당 window 분포:
window
2020_H1    21
2020_H2    18
2021_H2    12
2021_H1    11
2025_H1     9
2022_H1     8
2022_H2     7
2023_H1     7
2023_H2     6
2024_H2     4
2024_H1     3
Name: count, dtype: int64
해당 label_in 분포:
label_in
0    105
1      1
Name: count, dtype: int64

제거 후 shape: (2177, 11)
label_in 분포: {0: 2095, 1: 82}
편입 비율: 3.8%

=== 최종 학습 데이터 ===
       mktcap_rank  was_member  rank_momentum_1w  label_in
count      2177.00     2177.00           2177.00   2177.00
mean        115.38        0.85              0.02      0.04
std          74.91        0.36              4.62      0.19
min           1.00        0.00            -23.00      0.00
25%          52.00        1.00             -2.00      0.00
50%         107.00        1.00              0.00      0.00
75%         170.00        1.00              2.00      0.00
max         300.00        1.00             56

null 106개 중 편입(label_in=1)이 1개뿐이라 제거해도 학습에 거의 영향 없어요.

# 지금 피처가 3개인 이유
는 이게 진짜 베이스라인이기 때문이에요. 나중에 avg_mktcap, float_ratio, sector_mktcap_rank, within_90pct_rule 추가할 때 이 3개짜리 성능이 기준점이 돼요.

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

print("임포트 완료")

임포트 완료


# Walk-Forward CV 함수 정의

In [20]:
def walk_forward_cv(df, features, target, windows_ordered, min_train=2):
    """
    과거 window로 학습 → 다음 window로 테스트
    min_train: 최소 학습 window 수
    """
    results = []

    for i in range(min_train, len(windows_ordered)):
        train_windows = windows_ordered[:i]
        test_window   = windows_ordered[i]

        train = df[df['window'].isin(train_windows)]
        test  = df[df['window'] == test_window]

        # 테스트셋에 양성 없으면 스킵
        if test[target].sum() == 0:
            continue

        X_train = train[features].values
        y_train = train[target].values
        X_test  = test[features].values
        y_test  = test[target].values

        models = {
            'LogisticRegression': Pipeline([
                ('scaler', StandardScaler()),
                ('clf',    LogisticRegression(
                    class_weight='balanced',
                    max_iter=500,
                    random_state=42
                ))
            ]),
            'RandomForest': RandomForestClassifier(
                n_estimators=100,
                max_depth=5,
                class_weight='balanced',
                random_state=42
            ),
        }

        for name, model in models.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]

            tn, fp, fn, tp = confusion_matrix(
                y_test, y_pred, labels=[0, 1]
            ).ravel()

            results.append({
                'test_window' : test_window,
                'model'       : name,
                'AUC_ROC'     : round(roc_auc_score(y_test, y_prob), 4),
                'F1'          : round(f1_score(y_test, y_pred, zero_division=0), 4),
                'Precision'   : round(precision_score(y_test, y_pred, zero_division=0), 4),
                'Recall'      : round(recall_score(y_test, y_pred, zero_division=0), 4),
                'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                'n_pos_test'  : int(y_test.sum()),
                'train_size'  : len(y_train),
            })

    return pd.DataFrame(results)

# CV 실행

In [21]:
FEATURES = ['mktcap_rank', 'was_member', 'rank_momentum_1w']
TARGET   = 'label_in'

results = walk_forward_cv(
    df_clean,
    FEATURES,
    TARGET,
    WINDOWS_ORDERED,
    min_train=2
)

print("=== Walk-Forward CV 완료 ===")
print(f"총 테스트 반기 수: {results['test_window'].nunique()}")
print()

# 모델별 평균 성능
summary = (
    results
    .groupby('model')[['AUC_ROC','F1','Precision','Recall','TP','FP','FN']]
    .mean()
    .round(3)
)
print("=== 모델별 평균 성능 ===")
print(summary.to_string())
print()

# 반기별 상세 (LogisticRegression)
print("=== LogisticRegression 반기별 상세 ===")
lr_detail = results[results['model'] == 'LogisticRegression'][
    ['test_window','AUC_ROC','F1','Precision','Recall','TP','FP','FN','n_pos_test']
]
print(lr_detail.to_string(index=False))

=== Walk-Forward CV 완료 ===
총 테스트 반기 수: 9

=== 모델별 평균 성능 ===
                    AUC_ROC     F1  Precision  Recall     TP     FP     FN
model                                                                     
LogisticRegression    0.927  0.574      0.468   0.855  5.556  6.667  1.111
RandomForest          0.918  0.446      0.507   0.525  3.222  3.667  3.444

=== LogisticRegression 반기별 상세 ===
test_window  AUC_ROC     F1  Precision  Recall  TP  FP  FN  n_pos_test
    2021_H1   0.9716 0.5600     0.4118  0.8750   7  10   1           8
    2021_H2   0.9214 0.6400     0.5000  0.8889   8   8   1           9
    2022_H1   0.9763 0.6087     0.4375  1.0000   7   9   0           7
    2022_H2   0.9845 0.2222     0.1250  1.0000   1   7   0           1
    2023_H1   0.8052 0.8000     0.8000  0.8000   4   1   1           5
    2023_H2   0.9313 0.6667     0.5833  0.7778   7   5   2           9
    2024_H1   0.9674 0.6087     0.4375  1.0000   7   9   0           7
    2024_H2   0.8694 0.5000     0.363

✅ 결과 해석
- 좋은 것 : Recall 0.855는 실제 편입 종목 10개 중 8.5개를 잡는다는 뜻이에요. 놓치면 안 되는 편입 종목을 잘 찾아내고 있어요.
- 개선 여지 : FP(오탐)가 평균 6.7개예요. Precision이 0.47이니까 "편입될 것"으로 예측한 것 중 절반 정도만 실제 편입이에요. float_ratio, within_90pct_rule 같은 KRX 조건 피처 추가하면 줄어들 거예요.
- 2022_H2 이상치 : FP 7개에 TP 1개. 해당 반기에 시장 급변이 있었던 것 같아요. 나중에 확인해볼 포인트예요.

# 전체 데이터로 최종 모델 학습 (예측용)

In [23]:
X_all = df_clean[FEATURES].values
y_all = df_clean[TARGET].values

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=42
    ))
])
final_model.fit(X_all, y_all)

print("최종 모델 학습 완료")
print(f"학습 데이터: {len(X_all)}행 | 편입 {y_all.sum()}개 ({y_all.mean()*100:.1f}%)")

최종 모델 학습 완료
학습 데이터: 2177행 | 편입 82개 (3.8%)


# 2025_H2 예측 피처 생성

In [24]:
# 2025_H2 평가 기준일: 2025-12-05 (가장 가까운 날짜로)
PRED_WINDOW    = '2025_H2'
PRED_EVAL_DATE = '2025-12-05'

pred_closest = get_closest_date(PRED_EVAL_DATE)
print(f"2025_H2 스냅샷 기준일: {pred_closest.date()}")

# mktcap_rank 스냅샷
pred_snap = (
    weekly_clean[weekly_clean['date'] == pred_closest]
    [['ticker', 'mktcap_rank', 'mktcap', 'company']]
    .copy()
)

# rank_momentum_1w
pred_mom = (
    weekly_clean[weekly_clean['date'] == pred_closest]
    [['ticker', 'rank_momentum_1w']]
    .copy()
)

pred_df = pred_snap.merge(pred_mom, on='ticker', how='left')
pred_df['rank_momentum_1w'] = pred_df['rank_momentum_1w'].fillna(0)

# was_member: 2025_H1 기준 is_member
last_window_members = (
    labels[labels['window'] == '2025_H1']
    [['ticker', 'is_member']]
    .copy()
)
last_window_members['ticker'] = last_window_members['ticker'].astype(str)
last_window_members = last_window_members.rename(
    columns={'is_member': 'was_member'}
)

pred_df = pred_df.merge(last_window_members, on='ticker', how='left')
pred_df['was_member'] = pred_df['was_member'].fillna(0).astype(int)
pred_df['window'] = PRED_WINDOW

print(f"예측 대상 종목 수: {len(pred_df)}")
print(f"was_member=1 (직전 편입): {pred_df['was_member'].sum()}")
print(f"null 확인:\n{pred_df[FEATURES].isnull().sum()}")
print()
print(pred_df[['ticker','company','mktcap_rank','was_member','rank_momentum_1w']].head(5).to_string())

2025_H2 스냅샷 기준일: 2025-12-05
예측 대상 종목 수: 296
was_member=1 (직전 편입): 190
null 확인:
mktcap_rank         0
was_member          0
rank_momentum_1w    0
dtype: int64

   ticker      company  mktcap_rank  was_member  rank_momentum_1w
0     100         유한양행           63           1              -2.0
1  100090      SK오션플랜트          224           0              -8.0
2   10060       OCI홀딩스          171           1              -3.0
3  100840       SNT에너지          249           0              15.0
4   10120  LS ELECTRIC           44           1               1.0


# 예측 실행 

In [25]:
X_pred = pred_df[FEATURES].values

# 확률값 + 예측 라벨
pred_df['pred_in_prob'] = final_model.predict_proba(X_pred)[:, 1]
pred_df['pred_label']   = final_model.predict(X_pred)

# 확률 기준 순위
pred_df['pred_in_rank'] = pred_df['pred_in_prob'].rank(
    ascending=False, method='first'
).astype(int)

print("=== 2025_H2 예측 완료 ===")
print(f"편입 예측 종목 수 (pred_label=1): {pred_df['pred_label'].sum()}")
print()

# 상위 20개 출력
top20 = pred_df.nsmallest(20, 'pred_in_rank')[
    ['pred_in_rank','ticker','company','mktcap_rank',
     'was_member','rank_momentum_1w','pred_in_prob','pred_label']
]
print("=== 편입 확률 상위 20종목 ===")
print(top20.to_string(index=False))

=== 2025_H2 예측 완료 ===
편입 예측 종목 수 (pred_label=1): 106

=== 편입 확률 상위 20종목 ===
 pred_in_rank ticker    company  mktcap_rank  was_member  rank_momentum_1w  pred_in_prob  pred_label
            1   7810     코리아써키트          247           0              24.0      0.978185           1
            2  66575      LG전자우          244           0              17.0      0.971573           1
            3   3160        디아이          292           0               7.0      0.970586           1
            4 100840     SNT에너지          249           0              15.0      0.970503           1
            5   5880       대한해운          293           0               6.0      0.969743           1
            6  25540       한국단자          286           0               6.0      0.968326           1
            7  64960     SNT모티브          241           0              13.0      0.966681           1
            8    640   동아쏘시오홀딩스          269           0               7.0      0.965818           1
            9 3

In [26]:
# 전체 결과 저장
pred_df.to_csv('output/prediction_2025_H2.csv', index=False, encoding='utf-8-sig')

# 편입 예측 종목만 저장
pred_in = pred_df[pred_df['pred_label'] == 1].sort_values('pred_in_rank')
pred_in.to_csv('output/prediction_2025_H2_IN.csv', index=False, encoding='utf-8-sig')

print(f"저장 완료")
print(f"전체 예측: output/prediction_2025_H2.csv ({len(pred_df)}행)")
print(f"편입 예측: output/prediction_2025_H2_IN.csv ({len(pred_in)}행)")

저장 완료
전체 예측: output/prediction_2025_H2.csv (296행)
편입 예측: output/prediction_2025_H2_IN.csv (106행)


# 원인 진단

In [27]:
# 1. 실제 2025_H1 편입 종목 확인
h1_members = labels[
    (labels['window'] == '2025_H1') &
    (labels['is_member'] == 1)
]['ticker'].astype(str).tolist()

print(f"2025_H1 편입 종목 수: {len(h1_members)}")
print(f"예시: {h1_members[:10]}")
print()

# 2. pred_df에서 was_member 분포 확인
print("=== pred_df was_member 분포 ===")
print(pred_df['was_member'].value_counts())
print()

# 3. 학습 데이터에서 was_member와 label_in 관계 확인
print("=== 학습데이터 was_member vs label_in 크로스탭 ===")
print(pd.crosstab(df_clean['was_member'], df_clean['label_in'],
                  margins=True))
print()

# 4. 모델 계수 확인 (LogisticRegression)
lr_coef = final_model.named_steps['clf'].coef_[0]
for feat, coef in zip(FEATURES, lr_coef):
    print(f"  {feat:25s}: {coef:+.4f}")
print()

# 5. pred_df에서 was_member=1인 종목의 예측 확률 확인
print("=== was_member=1 종목 예측 확률 상위 10 ===")
print(pred_df[pred_df['was_member']==1]
      .nsmallest(10,'pred_in_rank')
      [['pred_in_rank','ticker','company','mktcap_rank',
        'was_member','pred_in_prob']]
      .to_string(index=False))

2025_H1 편입 종목 수: 200
예시: ['24110', '69960', '4020', '280360', '10120', '150', '2790', '8770', '11200', '375500']

=== pred_df was_member 분포 ===
was_member
1    190
0    106
Name: count, dtype: int64

=== 학습데이터 was_member vs label_in 크로스탭 ===
label_in       0   1   All
was_member                
0            250  73   323
1           1845   9  1854
All         2095  82  2177

  mktcap_rank              : +0.5053
  was_member               : -1.5263
  rank_momentum_1w         : +0.1659

=== was_member=1 종목 예측 확률 상위 10 ===
 pred_in_rank ticker company  mktcap_rank  was_member  pred_in_prob
          107   3620  KG모빌리티          281           1      0.280250
          108   1570      금양          291           1      0.258221
          109  39130    하나투어          270           1      0.251764
          110   2840    미원상사          280           1      0.244263
          111   5300    롯데칠성          212           1      0.232660
          112   5250  녹십자홀딩스          268           1      0.22963

## 근본 원인
- label_in의 정의가 "신규 편입"이에요. 기존 편입 종목(was_member=1)은 이미 안에 있으니까 label_in=0이에요. 그래서 모델이 was_member=0인 종목에 높은 확률을 부여하는 게 수학적으로 맞는 거예요.

즉,  타깃 변수를 잘못 골랐어요.
> label_in  = 신규 편입 (was_member=0 → 1)  ← 지금 쓰는 것
> is_member = 코스피200 현재 구성 여부       ← 써야 하는 것

2025년 12월 코스피200 구성 종목 200개 전체를 예측하려면 is_member를 타깃으로 써야 해요.

# 타깃 변수 수정 후 재학습 

In [28]:
# ── 타깃 변수 is_member로 변경
TARGET = 'is_member'

# ── Walk-Forward CV 재실행
results_v2 = walk_forward_cv(
    df_clean,
    FEATURES,
    TARGET,
    WINDOWS_ORDERED,
    min_train=2
)

# 모델별 평균 성능
summary_v2 = (
    results_v2
    .groupby('model')[['AUC_ROC','F1','Precision','Recall','TP','FP','FN']]
    .mean()
    .round(3)
)
print("=== 모델별 평균 성능 (타깃: is_member) ===")
print(summary_v2.to_string())
print()

# 계수 확인용 임시 모델
X_all = df_clean[FEATURES].values
y_all = df_clean[TARGET].values

final_model_v2 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=42
    ))
])
final_model_v2.fit(X_all, y_all)

print("=== 모델 계수 (is_member 기준) ===")
lr_coef = final_model_v2.named_steps['clf'].coef_[0]
for feat, coef in zip(FEATURES, lr_coef):
    direction = "↑ 편입 기여" if coef < 0 else "↓ 편입 기여"
    # mktcap_rank는 낮을수록 좋으므로 부호 해석 반대
    print(f"  {feat:25s}: {coef:+.4f}")

=== 모델별 평균 성능 (타깃: is_member) ===
                    AUC_ROC     F1  Precision  Recall     TP     FP     FN
model                                                                     
LogisticRegression    0.927  0.979      0.994   0.965  186.0  1.111  6.667
RandomForest          0.918  0.982      0.982   0.981  189.0  3.444  3.667

=== 모델 계수 (is_member 기준) ===
  mktcap_rank              : -0.5053
  was_member               : +1.5263
  rank_momentum_1w         : -0.1659


# 재예측 

In [29]:
X_pred = pred_df[FEATURES].values

pred_df['pred_in_prob'] = final_model_v2.predict_proba(X_pred)[:, 1]
pred_df['pred_label']   = final_model_v2.predict(X_pred)
pred_df['pred_in_rank'] = pred_df['pred_in_prob'].rank(
    ascending=False, method='first'
).astype(int)

print("=== 2025_H2 재예측 완료 ===")
print(f"편입 예측 종목 수: {pred_df['pred_label'].sum()}")
print()

top20 = pred_df.nsmallest(20, 'pred_in_rank')[
    ['pred_in_rank','ticker','company','mktcap_rank',
     'was_member','pred_in_prob','pred_label']
]
print("=== 편입 확률 상위 20종목 ===")
print(top20.to_string(index=False))

=== 2025_H2 재예측 완료 ===
편입 예측 종목 수: 190

=== 편입 확률 상위 20종목 ===
 pred_in_rank ticker   company  mktcap_rank  was_member  pred_in_prob  pred_label
            1   5930      삼성전자            1           1      0.953100           1
            2 329180   HD현대중공업            7           1      0.952894           1
            3    660    SK하이닉스            2           1      0.952797           1
            4 373220  LG에너지솔루션            3           1      0.952493           1
            5 207940  삼성바이오로직스            4           1      0.952187           1
            6  34020   두산에너빌리티            8           1      0.950943           1
            7 402340     SK스퀘어           14           1      0.950729           1
            8 105560      KB금융            9           1      0.950627           1
            9  35420     NAVER           15           1      0.950412           1
           10    270        기아           10           1      0.950310           1
           11  12450 한화에어로스페이스      

## 확인 필요: 편입 예측이 190개
코스피200은 정확히 200종목인데 190개만 예측됐어요. 그리고 확률값이 거의 다 0.95 근처로 몰려있어요. 이건 모델이 확률을 잘 분리 못하고 있다는 신호예요.원인 파악하고 후처리로 200개 맞출게요.

# 예측 확률 분포 확인 및 200개 보정

In [30]:
import matplotlib.pyplot as plt

# 1. 확률 분포 확인
print("=== 예측 확률 분포 ===")
print(pred_df['pred_in_prob'].describe().round(4))
print()

# 구간별 분포
bins = [0, 0.3, 0.5, 0.7, 0.9, 0.95, 0.97, 1.0]
pred_df['prob_bin'] = pd.cut(pred_df['pred_in_prob'], bins=bins)
print(pred_df['prob_bin'].value_counts().sort_index())
print()

# 2. was_member별 확률 분포
print("=== was_member별 평균 확률 ===")
print(pred_df.groupby('was_member')['pred_in_prob'].describe().round(4))
print()

# 3. 상위 200개로 강제 보정 (확률 순)
pred_df_top200 = pred_df.copy()
pred_df_top200['pred_label_200'] = 0
top200_idx = pred_df_top200.nsmallest(200, 'pred_in_rank').index
pred_df_top200.loc[top200_idx, 'pred_label_200'] = 1

print(f"=== 상위 200개 예측 결과 ===")
print(f"was_member=1 포함 수: {pred_df_top200[pred_df_top200['pred_label_200']==1]['was_member'].sum()}")
print(f"was_member=0 포함 수: {(pred_df_top200[pred_df_top200['pred_label_200']==1]['was_member']==0).sum()}")
print()

# 4. 190번째 ~ 210번째 경계 구간 확인 (어디서 잘리는지)
print("=== 순위 185~205 경계 구간 ===")
boundary = pred_df.sort_values('pred_in_rank').iloc[184:205][
    ['pred_in_rank','ticker','company','mktcap_rank',
     'was_member','pred_in_prob','pred_label']
]
print(boundary.to_string(index=False))

=== 예측 확률 분포 ===
count    296.0000
mean       0.5985
std        0.3995
min        0.0218
25%        0.0746
50%        0.8543
75%        0.9266
max        0.9531
Name: pred_in_prob, dtype: float64

prob_bin
(0.0, 0.3]      106
(0.3, 0.5]        0
(0.5, 0.7]        0
(0.7, 0.9]       83
(0.9, 0.95]      97
(0.95, 0.97]     10
(0.97, 1.0]       0
Name: count, dtype: int64

=== was_member별 평균 확률 ===
            count    mean     std     min     25%     50%     75%     max
was_member                                                               
0           106.0  0.0682  0.0322  0.0218  0.0449  0.0625  0.0825  0.2127
1           190.0  0.8943  0.0535  0.7198  0.8615  0.9125  0.9368  0.9531

=== 상위 200개 예측 결과 ===
was_member=1 포함 수: 190
was_member=0 포함 수: 10

=== 순위 185~205 경계 구간 ===
 pred_in_rank ticker   company  mktcap_rank  was_member  pred_in_prob  pred_label
          185   5250    녹십자홀딩스          268           1      0.770370           1
          186   5300      롯데칠성          212    

# 결과 해석 
was_member=1 (기존 편입 190개) → 확률 0.72~0.95  ← 전부 편입 예측
was_member=0 (신규 후보 106개) → 확률 0.02~0.21  ← 전부 미편입 예측

경계 구간을 보면 191위부터 was_member=0인데 삼성전자우(5위), 이수페타시스(59위), HD현대마린솔루션(67위) 같은 실제 유력 후보들이 포함돼 있어요. 이건 피처가 3개뿐이라 was_member 하나가 모든 것을 결정하기 때문이에요. 베이스라인의 한계가 여기서 드러나는 거예요.

# 최종 예측 200개 확정 및 저장

In [31]:
# 상위 200개를 최종 예측으로 확정
final_pred = pred_df.nsmallest(200, 'pred_in_rank').copy()
final_pred['final_label'] = 1

# 나머지는 0
pred_df_final = pred_df.copy()
pred_df_final['final_label'] = 0
pred_df_final.loc[final_pred.index, 'final_label'] = 1

print("=== 최종 예측 200종목 구성 ===")
print(f"was_member=1 (기존 편입 유지 예측): {final_pred['was_member'].sum()}개")
print(f"was_member=0 (신규 편입 예측):      {(final_pred['was_member']==0).sum()}개")
print()

# 신규 편입 예측 종목 (was_member=0) 확인
new_in = final_pred[final_pred['was_member']==0].sort_values('pred_in_rank')
print("=== 신규 편입 예측 종목 (was_member=0) ===")
print(new_in[['pred_in_rank','ticker','company',
              'mktcap_rank','pred_in_prob']].to_string(index=False))
print()

# 편출 예측 종목 (was_member=1이었는데 top200 밖)
churned = pred_df_final[
    (pred_df_final['was_member']==1) &
    (pred_df_final['final_label']==0)
].sort_values('pred_in_rank')
print("=== 편출 예측 종목 (기존 편입 → 탈락 예측) ===")
print(churned[['pred_in_rank','ticker','company',
               'mktcap_rank','pred_in_prob']].to_string(index=False))
print()

# 저장
final_pred.sort_values('pred_in_rank').to_csv(
    'output/prediction_2025_H2_final200.csv',
    index=False, encoding='utf-8-sig'
)
pred_df_final.to_csv(
    'output/prediction_2025_H2_all.csv',
    index=False, encoding='utf-8-sig'
)
print("저장 완료: output/prediction_2025_H2_final200.csv")
print("저장 완료: output/prediction_2025_H2_all.csv")

=== 최종 예측 200종목 구성 ===
was_member=1 (기존 편입 유지 예측): 190개
was_member=0 (신규 편입 예측):      10개

=== 신규 편입 예측 종목 (was_member=0) ===
 pred_in_rank ticker   company  mktcap_rank  pred_in_prob
          191   5935     삼성전자우            5      0.212662
          192   7660    이수페타시스           59      0.157990
          193 443060 HD현대마린솔루션           67      0.141970
          194   5387    현대차2우B           76      0.134733
          195  88980    맥쿼리인프라           90      0.132113
          196  64400    LG씨엔에스           84      0.128564
          197 267270    HD건설기계          101      0.123834
          198  62040      산일전기          103      0.122378
          199   3690      코리안리          159      0.116619
          200  82740      한화엔진          119      0.114848

=== 편출 예측 종목 (기존 편입 → 탈락 예측) ===
Empty DataFrame
Columns: [pred_in_rank, ticker, company, mktcap_rank, pred_in_prob]
Index: []

저장 완료: output/prediction_2025_H2_final200.csv
저장 완료: output/prediction_2025_H2_all.csv


# 우선주 제거 후 재예측 

In [32]:
# STOCK_META의 is_not_common 역할
# 지금은 해당 컬럼이 없으므로 ticker 패턴으로 필터링
# 우선주 ticker 특징: 끝자리가 5 (보통주 0, 우선주 5)
# 예: 삼성전자 005930(보통주), 삼성전자우 005935(우선주)

# 방법 1: 개별종목 파일의 '보(0)/우(1)' 컬럼 활용
pref_map = (
    indiv[['종목코드', '보(0)/우(1)']]
    .drop_duplicates('종목코드')
    .rename(columns={'종목코드': 'ticker', '보(0)/우(1)': 'is_preferred'})
)
pref_map['ticker'] = pref_map['ticker'].astype(str)

print("보통주/우선주 분포:")
print(pref_map['is_preferred'].value_counts())
print()

# pred_df에 붙이기
pred_df_filtered = pred_df.copy()
pred_df_filtered = pred_df_filtered.merge(pref_map, on='ticker', how='left')

# 우선주 여부 확인
print("pred_df 내 우선주 수:", (pred_df_filtered['is_preferred'] == 1).sum())
print("우선주 목록:")
print(pred_df_filtered[pred_df_filtered['is_preferred'] == 1][
    ['ticker', 'company', 'mktcap_rank', 'pred_in_rank']
].to_string(index=False))
print()

# 우선주 제거
pred_df_filtered = pred_df_filtered[
    pred_df_filtered['is_preferred'] != 1
].copy()

# pred_in_rank 재산정
pred_df_filtered['pred_in_rank'] = pred_df_filtered['pred_in_prob'].rank(
    ascending=False, method='first'
).astype(int)

print(f"필터링 후 예측 대상 종목 수: {len(pred_df_filtered)}")
print()

# 상위 200개 재확정
final_pred_v2 = pred_df_filtered.nsmallest(200, 'pred_in_rank').copy()

print("=== 필터링 후 최종 200종목 구성 ===")
print(f"was_member=1: {final_pred_v2['was_member'].sum()}개")
print(f"was_member=0: {(final_pred_v2['was_member']==0).sum()}개")
print()

new_in_v2 = final_pred_v2[final_pred_v2['was_member']==0].sort_values('pred_in_rank')
print("=== 수정된 신규 편입 예측 종목 ===")
print(new_in_v2[['pred_in_rank','ticker','company',
                  'mktcap_rank','pred_in_prob']].to_string(index=False))

보통주/우선주 분포:
is_preferred
0    852
1     90
Name: count, dtype: int64

pred_df 내 우선주 수: 8
우선주 목록:
ticker company  mktcap_rank  pred_in_rank
   155     두산우          160           211
 51915   LG화학우          194           221
  5385    현대차우           97           203
  5387  현대차2우B           76           194
  5935   삼성전자우            5           191
 66575   LG전자우          244           295
 71055 한국금융지주우          273           285
   815   삼성화재우          218           245

필터링 후 예측 대상 종목 수: 288

=== 필터링 후 최종 200종목 구성 ===
was_member=1: 190개
was_member=0: 10개

=== 수정된 신규 편입 예측 종목 ===
 pred_in_rank ticker   company  mktcap_rank  pred_in_prob
          191   7660    이수페타시스           59      0.157990
          192 443060 HD현대마린솔루션           67      0.141970
          193  88980    맥쿼리인프라           90      0.132113
          194  64400    LG씨엔에스           84      0.128564
          195 267270    HD건설기계          101      0.123834
          196  62040      산일전기          103      0.122378
       

In [33]:
# labels ticker도 str로 통일
pref_map_labels = pref_map.copy()
labels_tickers = labels.copy()
labels_tickers['ticker'] = labels_tickers['ticker'].astype(str)

# 우선주 ticker 목록
preferred_tickers = set(
    pref_map[pref_map['is_preferred'] == 1]['ticker'].tolist()
)
print(f"우선주 ticker 수: {len(preferred_tickers)}")

# df_clean에서 우선주 제거
df_clean_v2 = df_clean[
    ~df_clean['ticker'].astype(str).isin(preferred_tickers)
].copy()

print(f"학습 데이터: {len(df_clean)} → {len(df_clean_v2)} (우선주 {len(df_clean)-len(df_clean_v2)}행 제거)")
print(f"is_member 분포: {df_clean_v2['is_member'].value_counts().to_dict()}")
print()

# 재학습
X_all_v2 = df_clean_v2[FEATURES].values
y_all_v2  = df_clean_v2[TARGET].values

final_model_v3 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=42
    ))
])
final_model_v3.fit(X_all_v2, y_all_v2)
print("우선주 제거 후 재학습 완료")

# Walk-Forward CV도 재실행
results_v3 = walk_forward_cv(
    df_clean_v2, FEATURES, TARGET, WINDOWS_ORDERED, min_train=2
)
summary_v3 = (
    results_v3
    .groupby('model')[['AUC_ROC','F1','Precision','Recall','TP','FP','FN']]
    .mean()
    .round(3)
)
print()
print("=== 우선주 제거 후 모델 성능 ===")
print(summary_v3.to_string())

우선주 ticker 수: 90
학습 데이터: 2177 → 2177 (우선주 0행 제거)
is_member 분포: {1: 2095, 0: 82}

우선주 제거 후 재학습 완료

=== 우선주 제거 후 모델 성능 ===
                    AUC_ROC     F1  Precision  Recall     TP     FP     FN
model                                                                     
LogisticRegression    0.927  0.979      0.994   0.965  186.0  1.111  6.667
RandomForest          0.918  0.982      0.982   0.981  189.0  3.444  3.667


In [34]:
# ── 문제 1: ticker 형식 통일 확인
print("=== ticker 형식 비교 ===")
print("df_clean ticker 예시:", df_clean['ticker'].head(5).tolist())
print("pref_map ticker 예시:", pref_map['ticker'].head(5).tolist())
print()

# df_clean ticker를 lstrip('0') 후 비교
df_clean['ticker_str'] = df_clean['ticker'].astype(str).str.lstrip('0')
pref_map['ticker_str'] = pref_map['ticker'].astype(str).str.lstrip('0')

preferred_tickers_str = set(
    pref_map[pref_map['is_preferred'] == 1]['ticker_str'].tolist()
)
print(f"우선주 ticker 수: {len(preferred_tickers_str)}")
print(f"예시: {list(preferred_tickers_str)[:5]}")
print()

df_clean_v2 = df_clean[
    ~df_clean['ticker_str'].isin(preferred_tickers_str)
].copy()

print(f"학습 데이터: {len(df_clean)} → {len(df_clean_v2)}")
print(f"제거된 행: {len(df_clean) - len(df_clean_v2)}")
print()

# ── 문제 2: 타깃 변수 재확인
print("=== 타깃 분포 확인 ===")
print("is_member:", df_clean_v2['is_member'].value_counts().to_dict())
print("label_in:",  df_clean_v2['label_in'].value_counts().to_dict())
print()
print("※ is_member는 96%가 1 → 편향 심함")
print("※ label_in은 3.8%가 1 → 너무 희소")
print()

# ── 올바른 타깃: is_member이지만 CV 지표를 다르게 봐야 함
# is_member=1(200개) vs is_member=0(100개) 비율 window별 확인
print("=== window별 is_member 분포 ===")
print(df_clean_v2.groupby('window')['is_member']
      .agg(['sum','count'])
      .assign(ratio=lambda x: (x['sum']/x['count']).round(3))
      .to_string())

=== ticker 형식 비교 ===
df_clean ticker 예시: ['100', '100', '100', '100', '100']
pref_map ticker 예시: ['95570', '6840', '27410', '282330', '138930']

우선주 ticker 수: 90
예시: ['5257', '5725', '9835', '4835', '1525']

학습 데이터: 2177 → 2177
제거된 행: 0

=== 타깃 분포 확인 ===
is_member: {1: 2095, 0: 82}
label_in: {0: 2095, 1: 82}

※ is_member는 96%가 1 → 편향 심함
※ label_in은 3.8%가 1 → 너무 희소

=== window별 is_member 분포 ===
         sum  count  ratio
window                    
2020_H1  179    190  0.942
2020_H2  182    193  0.943
2021_H1  189    197  0.959
2021_H2  188    197  0.954
2022_H1  193    200  0.965
2022_H2  193    194  0.995
2023_H1  193    198  0.975
2023_H2  194    203  0.956
2024_H1  197    204  0.966
2024_H2  196    201  0.975
2025_H1  191    200  0.955


In [35]:
# ── Step 1: weekly_clean에서 ticker ↔ company 매핑 추출
ticker_name_map = (
    weekly_clean[['ticker', 'company']]
    .drop_duplicates('ticker')
    .copy()
)
ticker_name_map['ticker'] = ticker_name_map['ticker'].astype(str)

print(f"ticker-company 매핑 수: {len(ticker_name_map)}")
print(ticker_name_map.head(5))
print()

# ── Step 2: 우선주 판별 조건
# 조건 A: ticker 끝자리 숫자가 5 (보통주 0, 우선주 5)
# 조건 B: company명 끝에 '우', '우B', 'K', 'L' 등 포함
ticker_name_map['ticker_ends5'] = (
    ticker_name_map['ticker'].str[-1] == '5'
)
ticker_name_map['name_ends_with_pref'] = (
    ticker_name_map['company'].str.endswith('우') |
    ticker_name_map['company'].str.endswith('우B') |
    ticker_name_map['company'].str.endswith('우C') |
    ticker_name_map['company'].str.contains('우선주', na=False)
)

# 두 조건 중 하나라도 해당 → 우선주로 판단
ticker_name_map['is_preferred'] = (
    ticker_name_map['ticker_ends5'] | ticker_name_map['name_ends_with_pref']
).astype(int)

print("=== 우선주 판별 결과 ===")
print(ticker_name_map['is_preferred'].value_counts())
print()
print("우선주로 판별된 종목 예시:")
print(ticker_name_map[ticker_name_map['is_preferred']==1]
      [['ticker','company','ticker_ends5','name_ends_with_pref']]
      .head(20).to_string(index=False))
print()

# ── Step 3: 오탐 확인 (보통주인데 잘못 걸린 것)
print("=== 조건 불일치 케이스 (한 조건만 해당) ===")
mismatch = ticker_name_map[
    ticker_name_map['ticker_ends5'] != ticker_name_map['name_ends_with_pref']
]
print(mismatch[['ticker','company','ticker_ends5','name_ends_with_pref']].to_string(index=False))

ticker-company 매핑 수: 491
       ticker  company
94        100     유한양행
19133  100090  SK오션플랜트
158     10060   OCI홀딩스
89490  100840   SNT에너지
175    101140    인바이오젠

=== 우선주 판별 결과 ===
is_preferred
0    477
1     14
Name: count, dtype: int64

우선주로 판별된 종목 예시:
ticker company  ticker_ends5  name_ends_with_pref
   155     두산우          True                 True
 19175   신풍제약우          True                 True
  3545   대신증권우          True                 True
 51905 LG생활건강우          True                 True
 51915   LG화학우          True                 True
  5385    현대차우          True                 True
  5387  현대차2우B         False                 True
  5389  현대차3우B         False                 True
  5935   삼성전자우          True                 True
  6405  삼성SDI우          True                 True
 66575   LG전자우          True                 True
 71055 한국금융지주우          True                 True
   815   삼성화재우          True                 True
 90435 아모레퍼시픽우          True                

# 우선주 제거 후 학습 데이터 정리 및 재학습

In [36]:
# ── 우선주 ticker 목록 확정
preferred_tickers_final = set(
    ticker_name_map[ticker_name_map['is_preferred'] == 1]['ticker'].tolist()
)
print(f"최종 우선주 ticker 수: {len(preferred_tickers_final)}")
print(f"목록: {sorted(preferred_tickers_final)}")
print()

# ── df_clean에서 우선주 제거
# df_clean ticker가 '100' 형식 → lstrip('0') 불필요, 그대로 str 비교
df_clean['ticker'] = df_clean['ticker'].astype(str)
df_clean_v2 = df_clean[
    ~df_clean['ticker'].isin(preferred_tickers_final)
].copy()

print(f"학습 데이터: {len(df_clean)} → {len(df_clean_v2)}")
print(f"제거된 행: {len(df_clean) - len(df_clean_v2)}")
print()

# ── pred_df_filtered에서도 우선주 제거 (이미 Cell 21에서 했지만 재확인)
pred_df_filtered['ticker'] = pred_df_filtered['ticker'].astype(str)
pred_df_v2 = pred_df_filtered[
    ~pred_df_filtered['ticker'].isin(preferred_tickers_final)
].copy()

print(f"예측 대상: {len(pred_df_filtered)} → {len(pred_df_v2)}")
print(f"제거된 우선주: {len(pred_df_filtered) - len(pred_df_v2)}")
print()

# ── is_member/label_in 확인
print("=== 정제된 학습 데이터 타깃 분포 ===")
print("is_member:", df_clean_v2['is_member'].value_counts().to_dict())
print("label_in: ", df_clean_v2['label_in'].value_counts().to_dict())

최종 우선주 ticker 수: 14
목록: ['155', '19175', '3545', '51905', '51915', '5385', '5387', '5389', '5935', '6405', '66575', '71055', '815', '90435']

학습 데이터: 2177 → 2177
제거된 행: 0

예측 대상: 288 → 288
제거된 우선주: 0

=== 정제된 학습 데이터 타깃 분포 ===
is_member: {1: 2095, 0: 82}
label_in:  {0: 2095, 1: 82}


# 타깃 확정 + Walk-Forward CV 지표 재설계

Walk-Forward CV는 시계열 데이터 전용 교차검증 방식입니다.
- 과거 데이터로 학습하고, 그다음 시점의 미래 데이터로 검증하는 과정을 여러 번 앞으로 밀면서 반복합니다.
- 시간 순서를 절대 섞지 않는다는 점

In [37]:
# ── 타깃: is_member (코스피200 구성 여부)
# ── 지표: Top-K Precision 추가
#    "모델이 뽑은 상위 N개 중 실제 편입 종목 비율"이 핵심 지표

TARGET = 'is_member'
FEATURES = ['mktcap_rank', 'was_member', 'rank_momentum_1w']

def walk_forward_cv_v2(df, features, target, windows_ordered,
                       min_train=2, top_k=200):
    results = []

    for i in range(min_train, len(windows_ordered)):
        train_windows = windows_ordered[:i]
        test_window   = windows_ordered[i]

        train = df[df['window'].isin(train_windows)]
        test  = df[df['window'] == test_window].copy()

        if test[target].sum() == 0:
            continue

        X_tr = train[features].values
        y_tr = train[target].values
        X_te = test[features].values
        y_te = test[target].values

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('clf',    LogisticRegression(
                class_weight='balanced',
                max_iter=500,
                random_state=42
            ))
        ])
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_te)[:, 1]

        # 확률 기준 상위 K개 선택
        k = min(top_k, len(y_te))
        top_k_idx = y_prob.argsort()[::-1][:k]
        top_k_pred = np.zeros(len(y_te), dtype=int)
        top_k_pred[top_k_idx] = 1

        # 실제 편입 종목 수
        n_actual = int(y_te.sum())

        # Top-K Precision: 상위 K개 중 실제 편입 비율
        topk_prec = top_k_pred[y_te == 1].sum() / k

        # Recall: 실제 편입 중 상위 K에 포함된 비율
        recall = top_k_pred[y_te == 1].sum() / n_actual if n_actual > 0 else 0

        # 정확히 맞춘 종목 수
        hit = int(top_k_pred[y_te == 1].sum())
        miss = n_actual - hit

        results.append({
            'test_window'  : test_window,
            'AUC_ROC'      : round(roc_auc_score(y_te, y_prob), 4),
            'Top200_Prec'  : round(topk_prec, 4),   # 핵심 지표
            'Recall'       : round(recall, 4),
            'Hit'          : hit,    # 상위 200 중 실제 편입
            'Miss'         : miss,   # 놓친 실제 편입
            'FP'           : int(k - hit),  # 상위 200 중 실제 미편입
            'n_actual'     : n_actual,
            'train_size'   : len(y_tr),
        })

    return pd.DataFrame(results)

# ── 실행
results_v2 = walk_forward_cv_v2(
    df_clean_v2, FEATURES, TARGET, WINDOWS_ORDERED,
    min_train=2, top_k=200
)

print("=== Walk-Forward CV 결과 (Top-200 기준) ===")
print(results_v2[[
    'test_window','AUC_ROC','Top200_Prec','Recall','Hit','Miss','FP','n_actual'
]].to_string(index=False))
print()
print("=== 평균 성능 ===")
print(results_v2[['AUC_ROC','Top200_Prec','Recall','Hit','Miss','FP']].mean().round(3))

=== Walk-Forward CV 결과 (Top-200 기준) ===
test_window  AUC_ROC  Top200_Prec  Recall  Hit  Miss  FP  n_actual
    2021_H1   0.9716       0.9594  1.0000  189     0   8       189
    2021_H2   0.9214       0.9543  1.0000  188     0   9       188
    2022_H1   0.9763       0.9650  1.0000  193     0   7       193
    2022_H2   0.9845       0.9948  1.0000  193     0   1       193
    2023_H1   0.8052       0.9747  1.0000  193     0   5       193
    2023_H2   0.9313       0.9700  1.0000  194     0   6       194
    2024_H1   0.9674       0.9700  0.9848  194     3   6       197
    2024_H2   0.8694       0.9800  1.0000  196     0   4       196
    2025_H1   0.9186       0.9550  1.0000  191     0   9       191

=== 평균 성능 ===
AUC_ROC          0.927
Top200_Prec      0.969
Recall           0.998
Hit            192.333
Miss             0.333
FP               6.111
dtype: float64


# 우선주 제거 문제 진단 및 수정

In [38]:
# ── 1. df_clean에 우선주가 실제로 있는지 확인
print("=== df_clean ticker 예시 ===")
print(df_clean['ticker'].head(20).tolist())
print()

# 우선주 목록과 교집합
overlap = set(df_clean['ticker'].astype(str)) & preferred_tickers_final
print(f"df_clean과 우선주 목록 교집합: {len(overlap)}개")
print(f"교집합 ticker: {overlap}")
print()

# ── 2. labels.csv에 우선주가 있는지 확인
labels['ticker'] = labels['ticker'].astype(str)
labels_pref = labels[labels['ticker'].isin(preferred_tickers_final)]
print(f"labels에 우선주 행 수: {len(labels_pref)}")
print(labels_pref[['window','ticker','is_member']].head(10))
print()

# ── 3. weekly_clean ticker와 labels ticker 형식 비교
print("=== ticker 형식 비교 ===")
print("weekly_clean:", sorted(weekly_clean['ticker'].unique())[:10])
print("labels:      ", sorted(labels['ticker'].unique())[:10])
print("pref_tickers:", sorted(list(preferred_tickers_final))[:10])

=== df_clean ticker 예시 ===
['100', '100', '100', '100', '100', '100', '100', '100', '100', '100', '100', '10060', '10060', '10060', '10060', '10060', '10060', '10060', '10060', '10060']

df_clean과 우선주 목록 교집합: 0개
교집합 ticker: set()

labels에 우선주 행 수: 0
Empty DataFrame
Columns: [window, ticker, is_member]
Index: []

=== ticker 형식 비교 ===
weekly_clean: ['100', '100090', '10060', '100840', '101140', '10120', '10130', '10140', '1020', '102460']
labels:       ['100', '10060', '10120', '10130', '10140', '103140', '1040', '105560', '105630', '1060']
pref_tickers: ['155', '19175', '3545', '51905', '51915', '5385', '5387', '5389', '5935', '6405']


명확해요. labels.csv에 우선주가 애초에 없어요. 즉 학습 데이터는 이미 보통주만 있는 상태예요. 우선주 필터는 예측 대상(pred_df)에만 적용하면 돼요.

# 예측 대상 우선주 제거 확인 및 최종 예측

In [39]:
# ── 1. pred_df에서 우선주 제거 확인
pred_df['ticker'] = pred_df['ticker'].astype(str)

print("=== pred_df 우선주 포함 여부 ===")
pred_pref = pred_df[pred_df['ticker'].isin(preferred_tickers_final)]
print(f"우선주 행 수: {len(pred_pref)}")
print(pred_pref[['ticker','company','mktcap_rank']].to_string(index=False))
print()

# ── 2. 우선주 제거
pred_df_clean = pred_df[
    ~pred_df['ticker'].isin(preferred_tickers_final)
].copy()

print(f"예측 대상: {len(pred_df)} → {len(pred_df_clean)} (우선주 {len(pred_df)-len(pred_df_clean)}개 제거)")
print()

# ── 3. 최종 모델 학습 (우선주 없는 df_clean 그대로 사용)
X_all = df_clean[FEATURES].values
y_all = df_clean[TARGET].values

final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=42
    ))
])
final_model.fit(X_all, y_all)
print("최종 모델 학습 완료")
print(f"학습: {len(X_all)}행 | is_member=1: {y_all.sum()}개")
print()

# ── 4. 최종 예측
X_pred = pred_df_clean[FEATURES].values

pred_df_clean['pred_in_prob'] = final_model.predict_proba(X_pred)[:, 1]
pred_df_clean['pred_in_rank'] = pred_df_clean['pred_in_prob'].rank(
    ascending=False, method='first'
).astype(int)

# 상위 200개 확정
final_200 = pred_df_clean.nsmallest(200, 'pred_in_rank').copy()
final_200['final_label'] = 1

print("=== 최종 예측 200종목 구성 ===")
print(f"was_member=1 (기존 편입 유지): {final_200['was_member'].sum()}개")
print(f"was_member=0 (신규 편입 예측): {(final_200['was_member']==0).sum()}개")
print()

# ── 5. 신규 편입 예측 종목
new_in = final_200[final_200['was_member']==0].sort_values('pred_in_rank')
print("=== 신규 편입 예측 종목 ===")
print(new_in[['pred_in_rank','ticker','company',
              'mktcap_rank','pred_in_prob']].to_string(index=False))
print()

# ── 6. 기존 편입이었는데 200위 밖으로 밀린 종목 (편출 예측)
was_members = set(pred_df_clean[pred_df_clean['was_member']==1]['ticker'])
final_tickers = set(final_200['ticker'])
churned = pred_df_clean[
    pred_df_clean['ticker'].isin(was_members - final_tickers)
].sort_values('pred_in_rank')
print("=== 편출 예측 종목 (기존 편입 → 탈락) ===")
if len(churned) == 0:
    print("없음")
else:
    print(churned[['pred_in_rank','ticker','company',
                    'mktcap_rank','pred_in_prob']].to_string(index=False))

=== pred_df 우선주 포함 여부 ===
우선주 행 수: 8
ticker company  mktcap_rank
   155     두산우          160
 51915   LG화학우          194
  5385    현대차우           97
  5387  현대차2우B           76
  5935   삼성전자우            5
 66575   LG전자우          244
 71055 한국금융지주우          273
   815   삼성화재우          218

예측 대상: 296 → 288 (우선주 8개 제거)

최종 모델 학습 완료
학습: 2177행 | is_member=1: 2095개

=== 최종 예측 200종목 구성 ===
was_member=1 (기존 편입 유지): 190개
was_member=0 (신규 편입 예측): 10개

=== 신규 편입 예측 종목 ===
 pred_in_rank ticker   company  mktcap_rank  pred_in_prob
          191   7660    이수페타시스           59      0.157990
          192 443060 HD현대마린솔루션           67      0.141970
          193  88980    맥쿼리인프라           90      0.132113
          194  64400    LG씨엔에스           84      0.128564
          195 267270    HD건설기계          101      0.123834
          196  62040      산일전기          103      0.122378
          197   3690      코리안리          159      0.116619
          198  82740      한화엔진          119      0.114848
          1

In [40]:
# 전체 결과
pred_df_clean.sort_values('pred_in_rank').to_csv(
    'output/prediction_2025_H2_all.csv',
    index=False, encoding='utf-8-sig'
)

# 최종 200종목
final_200.sort_values('pred_in_rank').to_csv(
    'output/prediction_2025_H2_final200.csv',
    index=False, encoding='utf-8-sig'
)

print("저장 완료")
print(f"  전체: output/prediction_2025_H2_all.csv ({len(pred_df_clean)}행)")
print(f"  최종: output/prediction_2025_H2_final200.csv (200행)")

저장 완료
  전체: output/prediction_2025_H2_all.csv (288행)
  최종: output/prediction_2025_H2_final200.csv (200행)


# 🎉 베이스라인 MVP 완료

- 데이터: 주간시총300.xlsx + labels.csv 2177행, 피처 3개, 우선주 8개 제거
- 모델: Logistic Regression (Walk-Forward CV 9개 반기)
## 베이스라인 성능

| 지표 | 값 | 설명 |
|---|---:|---|
| AUC-ROC | 0.927 | 전체 분류 성능 |
| Top-200 Precision | 0.969 | 예측 상위 200종목 기준 정밀도 |
| Recall | 0.998 | 실제 편입 종목 재현율 |
| 평균 Hit | 192.3 / 200 | 예측 200종목 중 실제 적중 평균 개수 |
| 평균 Miss | 0.3개 | 실제 편입 종목 중 놓친 평균 개수 |
| 평균 FP | 6.1개 | 실제 미편입 종목을 편입으로 잘못 예측한 평균 개수 |

## 2025_H2 예측 결과

| 항목 | 값 |
|---|---:|
| 기존 편입 유지 | 190 |
| 신규 편입 예측 | 10 |
| 총 예측 종목 수 | 200 |

현재 베이스라인에서 사용한 피처는 3개예요.

## 현재 사용한 피처

| 피처 | 설명 | 소스 | 계산 방법 |
|---|---|---|---|
| `mktcap_rank` | 해당 반기 평가 기준일 시점 시총 순위 | `WEEKLY_SAMPLE.mktcap_rank` | 평가 기준일 날짜로 스냅샷 |
| `was_member` | 직전 반기 코스피200 편입 여부 | `LABELS.is_member` | 1 window shift |
| `rank_momentum_1w` | 1주 전 대비 시총 순위 변화 (양수=상승) | `WEEKLY_SAMPLE.mktcap_rank` | `rank(t-1) - rank(t)` |

딕셔너리 기준으로 보면 `FEATURE_KRX`에 설계된 13개 피처 중 3개만 쓴 상태예요.

아직 안 쓴 피처들이에요.

## 아직 사용하지 않은 피처

| 피처 | 필요 데이터 | 현황 |
|---|---|---|
| `avg_mktcap` | `WEEKLY_SAMPLE.mktcap` | 즉시 계산 가능 |
| `avg_trading_value` | `WEEKLY_SAMPLE.trading_value` | 2025-05 이후만 있음 |
| `avg_volume` | `WEEKLY_SAMPLE.volume` | 즉시 계산 가능 |
| `rank_momentum_4w` | `WEEKLY_SAMPLE.mktcap_rank` | 즉시 계산 가능 |
| `float_ratio` | `SHARES_LOCK` | 미수집 |
| `sector_mktcap_rank` | `SECTOR_MAP` | 커버리지 확인 필요 |
| `sector_trading_rank_pct` | `SECTOR_MAP + trading_value` | 미수집 |
| `within_90pct_rule` | `WEEKLY_SAMPLE.mktcap_rank` | 즉시 계산 가능 |
| `within_110pct_rule` | `WEEKLY_SAMPLE.mktcap_rank` | 즉시 계산 가능 |
| `is_existing_member` | `LABELS.is_member` | 즉시 계산 가능 (`was_member`와 동일) |
| `sector_member_count` | `SECTOR_MAP` | 커버리지 확인 필요 |

지금 FP 6개가 나오는 근본 원인은 신규 편입 종목 구분력 부족이에요. 신규 10개의 확률이 0.11~0.16으로 낮고 구분이 안 돼요. 아래 피처 추가 순서대로 가면 FP가 줄어들어요.


1순위 — 즉시 가능 (데이터 있음)
avg_mktcap : weekly_clean.mktcap 26주 rolling mean
rank_momentum_4w : 4주 전 대비 순위 변화
2순위 — SHARES_LOCK 수집 후
float_ratio : 유동비율 → 맥쿼리인프라(리츠), 코리안리 같은 오탐 제거에 직접 효과
3순위 — SECTOR_MAP 정비 후
within_90pct_rule / within_110pct_rule : KRX 공식 조건